<a href="https://colab.research.google.com/github/NNwobi-354/Land-Degradation-Trajectories-GEE-ML/blob/main/Degradation_Trajectories_Nigeria_vs_Russia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Comparative Analysis of Land Degradation Trajectories (2000–2025)
## Case Studies: Nigerian Sahel vs. Russian Kalmyk Steppe

### 1. Study Overview
This notebook implements a machine learning and remote sensing framework to analyze divergent drivers of land degradation in two distinct dryland ecosystems. While both regions show significant "browning" trends, the underlying stressors—ranging from conflict density in Nigeria to hydro-climatic forcing in Russia—are fundamentally different.

### 2. Objectives
*   **Trend Detection**: Statistical identification of Significant Browning, Greening, or Stable trajectories using Mann-Kendall tests.
*   **Driver Attribution**: Using Random Forest and XGBoost to rank the importance of 13 environmental and socio-political variables.
*   **Physical Calibration**: Documenting the conversion of satellite-derived indices into real-world physical units.

### 3. How to Use This Notebook
1.  **Environment Setup**: Run the **Repository Setup** cell below to clone the necessary scripts and datasets from GitHub.
2.  **Dependencies**: The notebook will automatically check for and install required libraries (e.g., `pymannkendall`, `xgboost`).
3.  **Data Loading**: The scripts are configured to pull CSV data directly from the GitHub repository via raw URLs for maximum reproducibility.
4.  **Execution**: Run the sections sequentially (Section 4.1 through 4.4) to generate the figures and interpretation summaries.

In [ ]:
# ==============================================================================
# REPOSITORY SETUP: CLONING THE RESEARCH FRAMEWORK
# ==============================================================================

import os

# Define repository details
REPO_URL = "https://github.com/NNwobi-354/Land-Degradation-Trajectories-GEE-ML.git"
REPO_NAME = "Land-Degradation-Trajectories-GEE-ML"

# Clone the repository if it doesn't already exist
if not os.path.exists(REPO_NAME):
    print(f"Cloning repository: {REPO_NAME}...")
    !git clone {REPO_URL}
else:
    print(f"Repository {REPO_NAME} already exists. Pulling latest updates...")
    %cd {REPO_NAME}
    !git pull
    %cd ..

# Verify the data folder exists
data_path = os.path.join(REPO_NAME, 'data')
if os.path.exists(data_path):
    print(f"\nSUCCESS: Repository cloned. Data folder located at: {data_path}")
    print(f"Files available: {os.listdir(data_path)}")
else:
    print("\nWARNING: Repository cloned but 'data' folder not found. Please check repository structure.")

In [ ]:
# ==============================================================================
# SECTION 4.1: STATISTICAL DETECTION OF DESERTIFICATION TRAJECTORIES
# FIGURE 5: DONUT DISTRIBUTIONS | FIGURE 6: AREA MAGNITUDE DISTRIBUTIONS
# ==============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Library check for Mann-Kendall
try:
    import pymannkendall as mk
except ImportError:
    !pip install pymannkendall
    import pymannkendall as mk

def perform_trend_analysis(df, region_name):
    results = []
    cells = df['Grid_Cell'].unique()
    for cell in cells:
        series = df[df['Grid_Cell'] == cell].sort_values('Year')['NDVI'].values
        res = mk.original_test(series)
        trend_cat = 'Stable'
        if res.p < 0.05:
            if res.trend == 'increasing': trend_cat = 'Significant Greening'
            elif res.trend == 'decreasing': trend_cat = 'Significant Browning'
        results.append({'Grid_Cell': cell, 'Trend': trend_cat, 'Slope': res.slope, 'Region': region_name})
    return pd.DataFrame(results)

# 1. DATA LOADING (Using GitHub Raw URLs)
# URLs point to the Land-Degradation-Trajectories-GEE-ML repository
ni_url = 'https://raw.githubusercontent.com/NNwobi-354/Land-Degradation-Trajectories-GEE-ML/main/data/Nigeria_Long_Format.csv'
ru_url = 'https://raw.githubusercontent.com/NNwobi-354/Land-Degradation-Trajectories-GEE-ML/main/data/Russia_Long_Format.csv'

ni_data = pd.read_csv(ni_url)
ru_data = pd.read_csv(ru_url)

# Proceed with Analysis
ni_results = perform_trend_analysis(ni_data, "Nigeria Sahel")
ru_results = perform_trend_analysis(ru_data, "Russian Steppe")
combined_results = pd.concat([ni_results, ru_results])

# ------------------------------------------------------------------------------
# FIGURE 5: DONUT PANEL (Earth-Tone Palette) - LEGEND ON RIGHT
# ------------------------------------------------------------------------------
plt.rcParams.update({'font.weight': 'bold', 'axes.labelweight': 'bold'})
fig5, axes5 = plt.subplots(1, 2, figsize=(20, 10), facecolor='white')

# Updated Earth-Tone Palette: Brown (Browning), Deep Green (Greening), Grey (Stable)
colors_map = {'Significant Greening': '#006400', 'Stable': '#A9A9A9', 'Significant Browning': '#8B4513'}

for i, (res, title) in enumerate([(ni_results, "NIGERIA SAHEL"), (ru_results, "RUSSIAN STEPPE")]):
    data = res['Trend'].value_counts().reindex(['Significant Greening', 'Stable', 'Significant Browning']).fillna(0)

    # Donut slices
    wedges, texts, autotexts = axes5[i].pie(
        data, autopct='%1.1f%%', startangle=140,
        colors=[colors_map[k] for k in data.index],
        pctdistance=0.82, explode=(0.05, 0, 0.05),
        textprops={'fontsize': 18, 'color': 'white', 'weight': 'bold'}
    )

    # White Center Circle for Donut Effect
    centre_circle = plt.Circle((0,0), 0.65, fc='white', linewidth=2, edgecolor='black')
    axes5[i].add_artist(centre_circle)

    # Central Label
    axes5[i].text(0, 0, f"{title}", ha='center', va='center', fontsize=22, weight='bold')

# Legend on the far right
fig5.legend(wedges, data.index, loc="center left", bbox_to_anchor=(0.92, 0.5),
            fontsize=16, frameon=True, title="Trend Categories", title_fontsize=17)

plt.tight_layout(rect=[0, 0, 0.9, 1])
plt.savefig('Figure_5_Donut_Trends_HighRes.png', dpi=700, bbox_inches='tight')
plt.show()

# ------------------------------------------------------------------------------
# FIGURE 6: CUMULATIVE MAGNITUDE AREA PLOT (High Density)
# ------------------------------------------------------------------------------
plt.figure(figsize=(15, 8))

# KDE Area Plot for Nigeria
sns.kdeplot(data=ni_results, x='Slope', fill=True, color='#3498db',
            label='Nigeria Sahel', alpha=0.4, linewidth=4)

# KDE Area Plot for Russia
sns.kdeplot(data=ru_results, x='Slope', fill=True, color='#e67e22',
            label='Russian Steppe', alpha=0.4, linewidth=4)

# Threshold line at zero
plt.axvline(0, color='black', linestyle='--', linewidth=3)

# Style & Annotation
plt.ylabel("DISTRIBUTION DENSITY", fontsize=18, weight='bold')
plt.xlabel("NDVI ANNUAL RATE OF CHANGE (THEIL-SEN SLOPE)", fontsize=18, weight='bold')
plt.xticks(fontsize=15, weight='bold')
plt.yticks(fontsize=15)
plt.grid(axis='both', linestyle=':', alpha=0.5)

# Legend on the right
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize=16, frameon=True)

plt.tight_layout()
plt.savefig('Figure_6_Area_Magnitude_HighRes.png', dpi=700, bbox_inches='tight')
plt.show()

# ------------------------------------------------------------------------------
# OUTPUT SUMMARY
# ------------------------------------------------------------------------------
print("\n" + "="*60)
print("SECTION 4.1 INTERPRETATION SUMMARY")
print("="*60)
print(f"NIGERIA SAHEL TREND DISTRIBUTION:")
print(ni_results['Trend'].value_counts(normalize=True).map(lambda x: f"{x:.1%}"))
print(f"\nRUSSIAN STEPPE TREND DISTRIBUTION:")
print(ru_results['Trend'].value_counts(normalize=True).map(lambda x: f"{x:.1%}"))
print("="*60)

In [ ]:
# ==============================================================================
# SECTION 4.2: INDIVIDUAL MODEL PERFORMANCE VALIDATION
# FIGURE 7: OBSERVED VS. PREDICTED (HIGH-CONTRAST PUBLICATION DESIGN)
# ==============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# 1. DATA LOADING (Using GitHub Raw URLs)
ni_url = 'https://raw.githubusercontent.com/NNwobi-354/Land-Degradation-Trajectories-GEE-ML/main/data/Nigeria_Long_Format.csv'
ru_url = 'https://raw.githubusercontent.com/NNwobi-354/Land-Degradation-Trajectories-GEE-ML/main/data/Russia_Long_Format.csv'

ni_data = pd.read_csv(ni_url)
ru_data = pd.read_csv(ru_url)

features = ['Albedo', 'Conflict', 'Evap_mm', 'Humidity', 'PET_mm',
            'Rain_mm', 'Runoff_mm', 'SensHeat_Wm2', 'SoilM_mm',
            'Temp_C', 'Wind_ms', 'Year']

def run_model_predictions(df, model_type):
    X = df[features]
    y = df['NDVI']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    if model_type == 'RF':
        model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
    else:
        model = XGBRegressor(n_estimators=200, learning_rate=0.05, random_state=42)

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    metrics = {
        'r2': r2_score(y_test, preds),
        'rmse': np.sqrt(mean_squared_error(y_test, preds)),
        'mae': mean_absolute_error(y_test, preds)
    }
    return y_test, preds, metrics

# Execution
ni_test_rf, ni_pred_rf, ni_m_rf = run_model_predictions(ni_data, "RF")
ni_test_xg, ni_pred_xg, ni_m_xg = run_model_predictions(ni_data, "XG")
ru_test_rf, ru_pred_rf, ru_m_rf = run_model_predictions(ru_data, "RF")
ru_test_xg, ru_pred_xg, ru_m_xg = run_model_predictions(ru_data, "XG")

# ------------------------------------------------------------------------------
# FIGURE 7: 4-PANEL BOLD SCATTER (ABCD)
# ------------------------------------------------------------------------------
# Set global thickness for the plot
plt.rcParams.update({
    'font.weight': 'bold',
    'axes.labelweight': 'bold',
    'axes.linewidth': 2.5,  # Thicker plot borders
    'xtick.major.width': 2,
    'ytick.major.width': 2
})

fig7, axes = plt.subplots(2, 2, figsize=(22, 20), facecolor='white')

plot_configs = [
    (ni_test_rf, ni_pred_rf, ni_m_rf, "A: NIGERIA SAHEL | RANDOM FOREST", "#2980b9"),
    (ni_test_xg, ni_pred_xg, ni_m_xg, "B: NIGERIA SAHEL | XGBOOST", "#8e44ad"),
    (ru_test_rf, ru_pred_rf, ru_m_rf, "C: RUSSIAN STEPPE | RANDOM FOREST", "#d35400"),
    (ru_test_xg, ru_pred_xg, ru_m_xg, "D: RUSSIAN STEPPE | XGBOOST", "#c0392b")
]

for i, (test, pred, m, title, color) in enumerate(plot_configs):
    ax = axes[i // 2, i % 2]

    # BOLD SCATTER POINTS: Increased size and black edges for visibility
    ax.scatter(test, pred, s=120, color=color, alpha=0.7, edgecolors='black', linewidths=1.5, label='Observations')

    # 1:1 IDENTITY LINE: Very thick dashed line
    ax.plot([0, 0.8], [0, 0.8], color='black', linestyle='--', linewidth=4, label='1:1 Line')

    # STATS BOX: Moved to the RIGHT side
    stats_text = (f"R²: {m['r2']:.3f}\n"
                  f"RMSE: {m['rmse']:.4f}\n"
                  f"MAE: {m['mae']:.4f}")

    ax.text(0.96, 0.04, stats_text, transform=ax.transAxes, fontsize=20, weight='bold',
            verticalalignment='bottom', horizontalalignment='right',
            bbox=dict(boxstyle='round,pad=0.6', facecolor='white', alpha=1.0, edgecolor='black', linewidth=2))

    # BOLD AXIS LABELS
    ax.set_title(title, fontsize=24, pad=20, weight='bold')
    ax.set_xlabel("OBSERVED NDVI", fontsize=20, labelpad=15)
    ax.set_ylabel("PREDICTED NDVI", fontsize=20, labelpad=15)

    # Sync Scales and Grids
    ax.set_xlim(0, 0.8); ax.set_ylim(0, 0.8)
    ax.tick_params(labelsize=16)
    ax.grid(True, linestyle='--', alpha=0.5, linewidth=1.5)

plt.tight_layout(pad=6.0)
plt.savefig('Figure_7_Model_Validation_UltraRes.png', dpi=700, bbox_inches='tight')
plt.show()

# ------------------------------------------------------------------------------
# INTERPRETATION SUMMARY
# ------------------------------------------------------------------------------
print("\n" + "="*60)
print("SECTION 4.2 MODEL VALIDATION SUMMARY")
print("="*60)
print(f"NIGERIA SAHEL  | RF R²: {ni_m_rf['r2']:.4f} | XGB R²: {ni_m_xg['r2']:.4f}")
print(f"RUSSIAN STEPPE | RF R²: {ru_m_rf['r2']:.4f} | XGB R²: {ru_m_xg['r2']:.4f}")
print("="*60)

In [ ]:
# ==============================================================================
# SECTION 4.3: COMPARATIVE DRIVER IMPORTANCE (GLOBAL SCALE)
# FIGURE 8: ENHANCED DUAL-PANEL FEATURE IMPORTANCE (MDI)
# ==============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor

# 1. DATA LOADING (Using GitHub Raw URLs)
ni_url = 'https://raw.githubusercontent.com/NNwobi-354/Land-Degradation-Trajectories-GEE-ML/main/data/Nigeria_Long_Format.csv'
ru_url = 'https://raw.githubusercontent.com/NNwobi-354/Land-Degradation-Trajectories-GEE-ML/main/data/Russia_Long_Format.csv'

ni_data = pd.read_csv(ni_url)
ru_data = pd.read_csv(ru_url)

# 2. DEFINING SCIENTIFIC FEATURES
features = [
    'Albedo', 'Conflict', 'Evap_mm', 'Humidity', 'PET_mm',
    'Rain_mm', 'Runoff_mm', 'SensHeat_Wm2', 'SoilM_mm',
    'Temp_C', 'Wind_ms'
]

name_mapping = {
    'Albedo': 'Surface Albedo',
    'Conflict': 'Conflict Density',
    'Evap_mm': 'Evapotranspiration',
    'Humidity': 'Specific Humidity',
    'PET_mm': 'Potential Evapotranspiration',
    'Rain_mm': 'Total Precipitation',
    'Runoff_mm': 'Surface Runoff',
    'SensHeat_Wm2': 'Sensible Heat Flux',
    'SoilM_mm': 'Soil Moisture',
    'Temp_C': 'Air Temperature',
    'Wind_ms': 'Wind Speed'
}

def get_feature_importance(df):
    X = df[features]
    y = df['NDVI']
    # Higher estimators for stability in global ranking
    rf = RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1)
    rf.fit(X, y)
    importance = pd.DataFrame({
        'Feature': [name_mapping[f] for f in features],
        'Importance': rf.feature_importances_
    }).sort_values(by='Importance', ascending=False)
    return importance

# Calculate rankings
ni_importance = get_feature_importance(ni_data)
ru_importance = get_feature_importance(ru_data)

# ------------------------------------------------------------------------------
# FIGURE 8: HIGH-IMPACT BOLD BAR CHART
# ------------------------------------------------------------------------------
plt.rcParams.update({
    'font.weight': 'bold',
    'axes.labelweight': 'bold',
    'axes.linewidth': 2.5,
    'xtick.major.width': 2,
    'ytick.major.width': 2
})

fig8, axes = plt.subplots(1, 2, figsize=(22, 12), facecolor='white')

# Panel A: Nigeria (Teal/Green Palette)
sns.barplot(x='Importance', y='Feature', data=ni_importance, ax=axes[0],
            palette='GnBu_r', edgecolor='black', linewidth=2.5)
axes[0].set_title("A: NIGERIA SAHEL | DRIVER RANKING", fontsize=22, pad=25, weight='bold')

# Panel B: Russia (Inferno/Fire Palette)
sns.barplot(x='Importance', y='Feature', data=ru_importance, ax=axes[1],
            palette='YlOrRd_r', edgecolor='black', linewidth=2.5)
axes[1].set_title("B: RUSSIAN STEPPE | DRIVER RANKING", fontsize=22, pad=25, weight='bold')

# Universal Styling and Value Annotations
for ax, df in zip(axes, [ni_importance, ru_importance]):
    ax.set_xlabel("MEAN DECREASE IN IMPURITY (MDI)", fontsize=18, labelpad=15)
    ax.set_ylabel("ENVIRONMENTAL & SOCIAL PREDICTORS", fontsize=18, labelpad=15)
    ax.tick_params(labelsize=16)
    ax.grid(axis='x', linestyle='--', alpha=0.4, linewidth=1.5)

    # Add data labels at the end of each bar
    for i, p in enumerate(ax.patches):
        width = p.get_width()
        ax.text(width + 0.002, p.get_y() + p.get_height()/2,
                f'{width:.3f}', ha='left', va='center', fontsize=14, weight='bold')

# Remove redundant y-label for Russia
axes[1].set_ylabel("")

plt.tight_layout(pad=5.0)
plt.savefig('Figure_8_Driver_Importance_HighImpact.png', dpi=700, bbox_inches='tight')
plt.show()

# ------------------------------------------------------------------------------
# INTERPRETATION SUMMARY & CSV EXPORTS
# ------------------------------------------------------------------------------
print("\n" + "="*60)
print("SECTION 4.3 INTERPRETATION SUMMARY (GLOBAL IMPORTANCE)")
print("="*60)
print("TOP 5 DRIVERS - NIGERIA SAHEL:")
print(ni_importance.head(5).to_string(index=False))
print("-" * 30)
print("TOP 5 DRIVERS - RUSSIAN STEPPE:")
print(ru_importance.head(5).to_string(index=False))
print("="*60)

# Export for tables in current working directory
ni_importance.to_csv('Nigeria_Driver_Importance_Final.csv', index=False)
ru_importance.to_csv('Russia_Driver_Importance_Final.csv', index=False)

In [ ]:
# ==============================================================================
# SECTION 4.4: REGIME SIGNATURE ANALYSIS (RADAR & BI-DIRECTIONAL DESIGN)
# FIGURE 9: POLAR DIAGNOSTIC SIGNATURE | FIGURE 10: DIVERGING REGIME AXIS
# ==============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. EMBEDDED SHAP DATA (Directly from study results)
features = ['Conflict Density', 'Surface Albedo', 'Specific Humidity', 'Evapotranspiration',
            'Air Temperature', 'Potential Evapotranspiration', 'Soil Moisture',
            'Surface Runoff', 'Total Precipitation', 'Wind Speed', 'Sensible Heat Flux']

# Re-aligning Russia data to match the feature order of Nigeria for a fair Radar comparison
ni_vals = [0.073077, 0.007797, 0.006077, 0.004236, 0.003586, 0.002596, 0.002558, 0.002239, 0.002181, 0.001983, 0.001738]
ru_vals = [0.005128, 0.004578, 0.003858, 0.010535, 0.007298, 0.002702, 0.007225, 0.004449, 0.003014, 0.044898, 0.001511]

# ------------------------------------------------------------------------------
# FIGURE 9: RADAR "SIGNATURE" PLOT (The Shape of Degradation)
# ------------------------------------------------------------------------------
plt.rcParams.update({'font.weight': 'bold', 'axes.labelweight': 'bold'})

angles = np.linspace(0, 2 * np.pi, len(features), endpoint=False).tolist()
ni_vals_r = ni_vals + [ni_vals[0]]
ru_vals_r = ru_vals + [ru_vals[0]]
angles += angles[:1]

fig9 = plt.figure(figsize=(14, 14), facecolor='white')
ax = fig9.add_subplot(111, polar=True)

# Plot Nigeria Signature
ax.plot(angles, ni_vals_r, color='#1b9e77', linewidth=4, label='Nigeria Sahel Signature')
ax.fill(angles, ni_vals_r, color='#1b9e77', alpha=0.3)

# Plot Russia Signature
ax.plot(angles, ru_vals_r, color='#d95f02', linewidth=4, label='Russian Steppe Signature')
ax.fill(angles, ru_vals_r, color='#d95f02', alpha=0.3)

# Professional Formatting
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
ax.set_thetagrids(np.degrees(angles[:-1]), features, fontsize=14, weight='bold')

plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=14, frameon=True, shadow=True)

# Save to current working directory
plt.savefig('Figure_9_Radar_Signature.png', dpi=700, bbox_inches='tight')
plt.show()

# ------------------------------------------------------------------------------
# FIGURE 10: BI-DIRECTIONAL DIVERGING AXIS (The "Mirror" Comparison)
# ------------------------------------------------------------------------------
fig10, ax10 = plt.subplots(figsize=(16, 12), facecolor='white')

y_pos = np.arange(len(features))
# Negative values for Nigeria to push them to the left on the diverging axis
ax10.barh(y_pos, [-v for v in ni_vals], color='#16a085', edgecolor='black', linewidth=2, label='Nigeria Sahel')
ax10.barh(y_pos, ru_vals, color='#e67e22', edgecolor='black', linewidth=2, label='Russian Steppe')

# Formatting the Mirror Axis
ax10.axvline(0, color='black', linewidth=3)
ax10.set_yticks(y_pos)
ax10.set_yticklabels(features, fontsize=16, weight='bold', ha='center', x=0.5)
ax10.invert_yaxis()  # Display top to bottom

# Customizing X-axis to show absolute values on both sides of the center line
ax10.set_xticklabels([f"{abs(x):.2f}" for x in ax10.get_xticks()], fontsize=14, weight='bold')

plt.xlabel("SHAP INFLUENCE MAGNITUDE (NDVI IMPACT)", fontsize=16, weight='bold')
plt.grid(axis='x', linestyle='--', alpha=0.4)
plt.legend(fontsize=15, frameon=True, shadow=True)

# Adjust y-ticks to prevent overlap with bars
ax10.tick_params(axis='y', pad=220)

plt.tight_layout()
# Save to current working directory
plt.savefig('Figure_10_Mirror_Divergence.png', dpi=700, bbox_inches='tight')
plt.show()

# ------------------------------------------------------------------------------
# INTERPRETATION SUMMARY
# ------------------------------------------------------------------------------
print("\n" + "="*60)
print("SECTION 4.4 REGIME SIGNATURE SUMMARY")
print("="*60)
print("The 'Shape' of Nigeria's degradation is dominated by Social Conflict.")
print("The 'Shape' of Russia's degradation is dominated by Physical Wind Velocity.")
print("="*60)

In [ ]:
# ==============================================================================
# SECTION 4.5: SPATIO-TEMPORAL ANOMALIES AND CRISIS YEARS
# FIGURE 11: COMPARATIVE ANOMALY TIMELINE | FIGURE 12: INTERACTION BUBBLE PLOT
# ==============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. DATA LOADING (Using GitHub Raw URLs)
ni_url = 'https://raw.githubusercontent.com/NNwobi-354/Land-Degradation-Trajectories-GEE-ML/main/data/Nigeria_Long_Format.csv'
ru_url = 'https://raw.githubusercontent.com/NNwobi-354/Land-Degradation-Trajectories-GEE-ML/main/data/Russia_Long_Format.csv'

ni_data = pd.read_csv(ni_url)
ru_data = pd.read_csv(ru_url)

def detect_anomalies(df, region_name):
    yearly_stats = df.groupby('Year')['NDVI'].mean().reset_index()
    overall_mean = yearly_stats['NDVI'].mean()
    yearly_stats['Anomaly'] = yearly_stats['NDVI'] - overall_mean
    yearly_stats['Region'] = region_name
    return yearly_stats

ni_anom = detect_anomalies(ni_data, "Nigeria Sahel")
ru_anom = detect_anomalies(ru_data, "Russian Steppe")

# GLOBAL PLOT STYLING FOR MAXIMUM BOLDNESS
plt.rcParams.update({
    'font.weight': 'bold',
    'axes.labelweight': 'bold',
    'axes.titleweight': 'bold',
    'axes.linewidth': 2.5,        # Thicker plot borders
    'xtick.major.width': 2.0,     # Thicker tick marks
    'ytick.major.width': 2.0
})

# ------------------------------------------------------------------------------
# FIGURE 11: SYNCHRONIZED CRISIS TIMELINE (Ultra-Bold)
# ------------------------------------------------------------------------------
plt.figure(figsize=(18, 8), facecolor='white')

# Plotting Nigeria (Blue) - Increased linewidth and marker size
markerline, stemlines, baseline = plt.stem(ni_anom['Year'] - 0.15, ni_anom['Anomaly'],
                                          linefmt='blue', markerfmt='bo', label='Nigeria Sahel')
plt.setp(stemlines, 'linewidth', 3, 'alpha', 0.7)  # Heavier stem lines
plt.setp(markerline, 'markersize', 10)

# Plotting Russia (Red) - Increased linewidth and marker size
markerline2, stemlines2, baseline2 = plt.stem(ru_anom['Year'] + 0.15, ru_anom['Anomaly'],
                                             linefmt='red', markerfmt='ro', label='Russian Steppe')
plt.setp(stemlines2, 'linewidth', 3, 'alpha', 0.7) # Heavier stem lines
plt.setp(markerline2, 'markersize', 10)

plt.axhline(0, color='black', linewidth=3, linestyle='-') # Heavy baseline
plt.ylabel("NDVI ANOMALY (DEVIATION)", fontsize=16, weight='bold')
plt.xlabel("YEAR", fontsize=16, weight='bold')
plt.xticks(ni_anom['Year'], rotation=45, fontsize=14, weight='bold')
plt.yticks(fontsize=14, weight='bold')

# Legend placed on the right with a thick border
plt.legend(fontsize=14, loc='center left', bbox_to_anchor=(1, 0.5), frameon=True, edgecolor='black', shadow=True)
plt.grid(axis='y', alpha=0.4, linewidth=1.5, linestyle='--')

plt.tight_layout()
plt.savefig('Figure_11_Anomaly_Timeline_Bold.png', dpi=700, bbox_inches='tight')
plt.show()

# ------------------------------------------------------------------------------
# FIGURE 12: INTERACTION BUBBLE PLOT (Ultra-Bold)
# ------------------------------------------------------------------------------
plt.figure(figsize=(13, 9), facecolor='white')

# Aggregate data for interaction
inter_df = ru_data.copy()
inter_df['Wind_Bin'] = pd.qcut(inter_df['Wind_ms'], 6).apply(lambda x: x.mid)
inter_df['SoilM_Bin'] = pd.qcut(inter_df['SoilM_mm'], 6).apply(lambda x: x.mid)
bubble_data = inter_df.groupby(['Wind_Bin', 'SoilM_Bin'])['NDVI'].mean().reset_index()

# Normalize NDVI for bubble sizing
norm_ndvi = (bubble_data['NDVI'] - bubble_data['NDVI'].min()) / (bubble_data['NDVI'].max() - bubble_data['NDVI'].min())

scatter = plt.scatter(bubble_data['SoilM_Bin'], bubble_data['Wind_Bin'],
            s=norm_ndvi * 1200,          # Larger bubble scale
            c=bubble_data['NDVI'],       # Color represents NDVI intensity
            cmap='RdYlGn', alpha=0.8,
            edgecolors='black', linewidth=2.5) # Heavy bubble borders

plt.xlabel("SOIL MOISTURE (mm)", fontsize=16, weight='bold', labelpad=10)
plt.ylabel("WIND SPEED (m/s)", fontsize=16, weight='bold', labelpad=10)
plt.xticks(fontsize=14, weight='bold')
plt.yticks(fontsize=14, weight='bold')

# Colorbar with bold labels
cbar = plt.colorbar(scatter, fraction=0.046, pad=0.04)
cbar.set_label('Mean NDVI Signal Magnitude', fontweight='bold', fontsize=14, labelpad=15)
cbar.ax.tick_params(labelsize=12)

plt.grid(True, linestyle='--', alpha=0.5, linewidth=1.5)
plt.tight_layout()
plt.savefig('Figure_12_Interaction_BubblePlot_Bold.png', dpi=700, bbox_inches='tight')
plt.show()

# ------------------------------------------------------------------------------
# SUMMARY FOR INTERPRETATION
# ------------------------------------------------------------------------------
print("\n" + "="*60)
print("SECTION 4.5 INTERPRETATION SUMMARY")
print("="*60)
print(f"Nigeria Crisis Years: {ni_anom.nsmallest(3, 'Anomaly')['Year'].values.tolist()}")
print(f"Russia Crisis Years: {ru_anom.nsmallest(3, 'Anomaly')['Year'].values.tolist()}")
print("-" * 30)
print("Note: The 'Perfect Storm' zone is clearly visible where bubble size shrinks")
print("and color shifts toward Red (Top-Left quadrant of Figure 12).")
print("="*60)

In [ ]:
# ==============================================================================
# SECTION 4.6: STATISTICAL ROBUSTNESS AND PREDICTIVE RELIABILITY
# FIGURE 13: VIP STRUCTURAL SIGNIFICANCE | FIGURE 14: Z-SCORE EXTREMITY
# ==============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cross_decomposition import PLSRegression

# 1. DATA LOADING (Using GitHub Raw URLs)
ni_url = 'https://raw.githubusercontent.com/NNwobi-354/Land-Degradation-Trajectories-GEE-ML/main/data/Nigeria_Long_Format.csv'
ru_url = 'https://raw.githubusercontent.com/NNwobi-354/Land-Degradation-Trajectories-GEE-ML/main/data/Russia_Long_Format.csv'

ni_data = pd.read_csv(ni_url)
ru_data = pd.read_csv(ru_url)

def calculate_vip_robustness(df, features, target):
    X = df[features]
    y = df[target]
    # Standardize data to ensure fair statistical comparison
    X_std = (X - X.mean()) / X.std()
    y_std = (y - y.mean()) / y.std()

    pls = PLSRegression(n_components=2)
    pls.fit(X_std, y_std)

    # Calculate VIP (Variable Influence on Projection)
    # VIP > 1.0 is the international standard for statistical significance in PLS
    weights = pls.x_weights_
    vip = np.sqrt(len(features) * np.sum(weights**2, axis=1))
    return pd.DataFrame({'Feature': features, 'VIP_Score': vip}).sort_values('VIP_Score', ascending=True)

# Define the 11 parameters used in your study
features = ['Conflict', 'Albedo', 'Humidity', 'Evap_mm', 'Temp_C',
            'PET_mm', 'SoilM_mm', 'Runoff_mm', 'Rain_mm', 'Wind_ms', 'SensHeat_Wm2']

ni_vip = calculate_vip_robustness(ni_data, features, 'NDVI')
ru_vip = calculate_vip_robustness(ru_data, features, 'NDVI')

# GLOBAL STYLING FOR BOLDNESS
plt.rcParams.update({'font.weight': 'bold', 'axes.labelweight': 'bold', 'axes.linewidth': 2.5})

# ------------------------------------------------------------------------------
# FIGURE 13: COMPARATIVE VIP SIGNIFICANCE (Lollipop Design)
# ------------------------------------------------------------------------------
fig13, axes = plt.subplots(1, 2, figsize=(22, 11), facecolor='white')

for i, (data, title, color) in enumerate([
    (ni_vip, "A: NIGERIA SAHEL | STRUCTURAL SIGNIFICANCE", '#1b9e77'),
    (ru_vip, "B: RUSSIAN STEPPE | STRUCTURAL SIGNIFICANCE", '#d95f02')
]):
    # Create the Lollipop Plot
    axes[i].hlines(y=data['Feature'], xmin=0, xmax=data['VIP_Score'], color='black', alpha=0.6, linewidth=3)
    axes[i].scatter(data['VIP_Score'], data['Feature'], color=color, s=250, edgecolors='black', linewidth=2.5, zorder=3)

    # Highlight the "Statistical Significance Zone" (VIP > 1.0)
    axes[i].axvspan(1.0, data['VIP_Score'].max() + 0.2, color='gray', alpha=0.1, label='Significance Zone (VIP > 1)')
    axes[i].axvline(1.0, color='red', linestyle='--', linewidth=4)

    axes[i].set_title(title, fontsize=20, pad=20)
    axes[i].set_xlabel("VARIABLE INFLUENCE ON PROJECTION (VIP SCORE)", fontsize=15)
    axes[i].tick_params(labelsize=14)
    axes[i].grid(axis='x', linestyle=':', alpha=0.5)

plt.tight_layout()
plt.savefig('Figure_13_Statistical_VIP_Lollipop.png', dpi=700)
plt.show()

# ------------------------------------------------------------------------------
# FIGURE 14: Z-SCORE DISTRIBUTION (Statistical Anomaly Strength)
# ------------------------------------------------------------------------------
plt.figure(figsize=(15, 8), facecolor='white')

# Calculate Z-Scores to confirm "Crisis Years" are statistically extreme outliers
ni_z = (ni_data.groupby('Year')['NDVI'].mean() - ni_data['NDVI'].mean()) / ni_data['NDVI'].std()
ru_z = (ru_data.groupby('Year')['NDVI'].mean() - ru_data['NDVI'].mean()) / ru_data['NDVI'].std()

plt.hist(ni_z, bins=15, alpha=0.7, label='Nigeria Sahel NDVI Z-Dist', color='#16a085', edgecolor='black', linewidth=2)
plt.hist(ru_z, bins=15, alpha=0.7, label='Russian Steppe NDVI Z-Dist', color='#e67e22', edgecolor='black', linewidth=2)

# 95% Confidence interval for identifying "Severe" anomalies
plt.axvline(-1.96, color='black', linestyle=':', linewidth=4, label='95% Statistical Confidence (p < 0.05)')
plt.xlabel("Z-SCORE (DEVIATIONS FROM REGIONAL MEAN)", fontsize=16)
plt.ylabel("FREQUENCY (NUMBER OF YEARS)", fontsize=16)
plt.legend(fontsize=14, frameon=True, shadow=True)

plt.tight_layout()
plt.savefig('Figure_14_ZScore_Robustness.png', dpi=700)
plt.show()

# ------------------------------------------------------------------------------
# FINAL ROBUSTNESS SUMMARY OUTPUT
# ------------------------------------------------------------------------------
print("\n" + "="*60)
print("SECTION 4.6: STATISTICAL ROBUSTNESS RESULTS SUMMARY")
print("="*60)
print(f"NI - Significant Drivers (VIP > 1.0): {len(ni_vip[ni_vip['VIP_Score'] > 1])} of {len(features)}")
print(f"RU - Significant Drivers (VIP > 1.0): {len(ru_vip[ru_vip['VIP_Score'] > 1])} of {len(features)}")
print("-" * 30)
print("TOP STRUCTURAL DRIVER (MOST ROBUST):")
print(f"Nigeria Sahel: {ni_vip.iloc[-1]['Feature']} (VIP: {ni_vip.iloc[-1]['VIP_Score']:.3f})")
print(f"Russian Steppe: {ru_vip.iloc[-1]['Feature']} (VIP: {ru_vip.iloc[-1]['VIP_Score']:.3f})")
print("-" * 30)
print("STATISTICAL OUTLIER ANALYSIS (FIGURE 14):")
print(f"Nigeria - Years exceeding 95% threshold: {list(ni_z[ni_z < -1.96].index)}")
print(f"Russia - Years exceeding 95% threshold: {list(ru_z[ru_z < -1.96].index)}")
print("="*60)